In [1]:
import os

In [2]:
from google.cloud import storage
from urllib.parse import urlparse

In [6]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"google_drive_api.json"

In [7]:
client = storage.Client()


In [8]:

def download_gcs_file(gcs_link: str, destination_path: str):
    """
    Downloads a file from GCS using a service account.
    
    Supports:
    - gs://bucket/path/file
    - https://storage.googleapis.com/bucket/path/file
    """

    client = storage.Client()

    # Case 1: gs://bucket/path
    if gcs_link.startswith("gs://"):
        path = gcs_link.replace("gs://", "")
        bucket_name, blob_path = path.split("/", 1)

    # Case 2: https URL
    elif "storage.googleapis.com" in gcs_link:
        parsed = urlparse(gcs_link)
        parts = parsed.path.lstrip("/").split("/", 1)
        bucket_name = parts[0]
        blob_path = parts[1]

    else:
        raise ValueError("Unsupported GCS link format")

    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)

    # Create local folders if needed
    os.makedirs(os.path.dirname(destination_path), exist_ok=True)

    blob.download_to_filename(destination_path)

    print(f"Downloaded: {destination_path}")

In [9]:
gdrive_folder_link = "https://drive.google.com/drive/folders/1s0u_6xVuPHVnxgqiD99mBvncs21BJEA5?usp=drive_link"
dest_path = r"D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check"

In [ ]:
download_gcs_file(gdrive_folder_link,dest_path)

In [14]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2 import service_account
import io
import os

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
SERVICE_ACCOUNT_FILE = 'google_drive_api.json'


def download_drive_folder(folder_id, local_path):
    creds = service_account.Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE, scopes=SCOPES
    )

    service = build('drive', 'v3', credentials=creds)

    os.makedirs(local_path, exist_ok=True)

    def download_recursive(folder_id, current_path):
        results = service.files().list(
            q=f"'{folder_id}' in parents",
            fields="files(id, name, mimeType)"
        ).execute()

        items = results.get('files', [])

        for item in items:
            file_id = item['id']
            name = item['name']
            mime_type = item['mimeType']

            file_path = os.path.join(current_path, name)

            # If folder → recurse
            if mime_type == 'application/vnd.google-apps.folder':
                os.makedirs(file_path, exist_ok=True)
                download_recursive(file_id, file_path)

            else:
                request = service.files().get_media(fileId=file_id)
                fh = io.FileIO(file_path, 'wb')
                downloader = MediaIoBaseDownload(fh, request)

                done = False
                while not done:
                    status, done = downloader.next_chunk()

                print(f"Downloaded: {file_path}")

    download_recursive(folder_id, local_path)

In [15]:
folder_id = "1s0u_6xVuPHVnxgqiD99mBvncs21BJEA5"

download_drive_folder(folder_id, r"D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check")

Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\Engine Records\F101. Engine Manufacture Delivery Documents\ESN 577270 - Log Book.pdf
Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\Engine Records\F101. Engine Manufacture Delivery Documents\ESN 577270 - EDS.pdf
Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\Engine Records\F101. Engine Manufacture Delivery Documents\A ESN 726413 GMF Aeroasia InstantBSI R1.pdf


In [16]:
import re

def extract_drive_folder_id(link: str):
    match = re.search(r'/folders/([a-zA-Z0-9_-]+)', link)
    if match:
        return match.group(1)
    else:
        raise ValueError("Invalid Google Drive folder link")

In [17]:
print(extract_drive_folder_id("https://drive.google.com/drive/folders/1s0u_6xVuPHVnxgqiD99mBvncs21BJEA5?usp=drive_link"))

1s0u_6xVuPHVnxgqiD99mBvncs21BJEA5


In [21]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2 import service_account
import io
import os

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
SERVICE_ACCOUNT_FILE = 'google_drive_api.json'


def download_gdrive_folder_from_link(folder_link, local_path):
    
    def extract_drive_folder_id(link: str):
        match = re.search(r'/folders/([a-zA-Z0-9_-]+)', link)
        if match:
            return match.group(1)
        else:
            raise ValueError("Invalid Google Drive folder link")
    
    folder_id = extract_drive_folder_id(folder_link)
    
    creds = service_account.Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE, scopes=SCOPES
    )

    service = build('drive', 'v3', credentials=creds)

    os.makedirs(local_path, exist_ok=True)

    def download_recursive(folder_id, current_path):
        results = service.files().list(
            q=f"'{folder_id}' in parents",
            fields="files(id, name, mimeType)"
        ).execute()

        items = results.get('files', [])

        for item in items:
            file_id = item['id']
            name = item['name']
            mime_type = item['mimeType']

            file_path = os.path.join(current_path, name)

            # If folder → recurse
            if mime_type == 'application/vnd.google-apps.folder':
                os.makedirs(file_path, exist_ok=True)
                download_recursive(file_id, file_path)

            else:
                request = service.files().get_media(fileId=file_id)
                fh = io.FileIO(file_path, 'wb')
                downloader = MediaIoBaseDownload(fh, request)

                done = False
                while not done:
                    status, done = downloader.next_chunk()

                print(f"Downloaded: {file_path}")

    download_recursive(folder_id, local_path)

In [19]:
download_gdrive_folder_from_link("https://drive.google.com/drive/folders/1s0u_6xVuPHVnxgqiD99mBvncs21BJEA5?usp=drive_link",
r"D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check")

Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\Engine Records\F101. Engine Manufacture Delivery Documents\ESN 577270 - Log Book.pdf
Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\Engine Records\F101. Engine Manufacture Delivery Documents\ESN 577270 - EDS.pdf
Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\Engine Records\F101. Engine Manufacture Delivery Documents\A ESN 726413 GMF Aeroasia InstantBSI R1.pdf


In [ ]:
download_gdrive_folder_from_link("https://drive.google.com/drive/folders/19BaCjiM4GrNZnVBQZOSG4OIJ1nSvSDYo?usp=drive_link",
r"D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check")

Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\16. Last Borescope Inspection\Current BSI.pdf
Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\16. Last Borescope Inspection\ESN 569394 - BSI - OPEN CLOSE Task Card N451PE_1451_FL2351_TASK_10008.pdf
Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\16. Last Borescope Inspection\BSI_569394-01-25.pdf
Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\16. Last Borescope Inspection\F011 Last Borescope Report ESN 569394.pdf
Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\16. Last Borescope Inspection\2014-10-06_Boroscope_Inspection_Borescope_Report_0000.pdf
Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\16. Last Borescope Inspection\2014-10-06_AD_Status_AD_Status_(RFL)_0000.pdf
Downloaded: D:\Samuel.R\GEM RECORDS MANAGEMENT PORTAL\WGdrive_link_check\19. LLP Summary\Latest\Thumbs.db
Downloaded: D:\Samu

KeyboardInterrupt: 

: 